# Clasificación de sentimiento — Parte 1
## Experimento: especialista en opiniones contrapuestas

### Objetivo

Evaluar si un clasificador especializado puede mejorar las predicciones
del ensamble v3b en reseñas con dos opiniones de signo contrario.

### Hipótesis

El contexto textual de cada opinión podría aportar información para
determinar cuál coincide con la etiqueta de la reseña.

Utilizaremos representaciones clásicas TF-IDF y modelos de scikit-learn,
sin redes neuronales ni embeddings preentrenados.

### Evaluación

- Identificar los casos candidatos mediante reglas basadas en el texto.
- Ajustar las transformaciones aprendidas únicamente con entrenamiento
  dentro de cada fold.
- Comparar el especialista y el ensamble sobre las mismas reseñas.
- Medir la mejora global y los aciertos perdidos al incorporar el especialista.
- Mantener eval.csv fuera del diseño y selección de modelos.

Este experimento es exploratorio: utiliza un dataset que ya se ha
analizado en iteraciones anteriores.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

RANDOM_STATE = 42
SEMILLAS = [42, 2024]
CLASES = np.array(["negativo", "neutral", "positivo"])

ROOT = Path.cwd()
if not (ROOT / "data" / "train.csv").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
NOTEBOOK_ORIGINAL = (
    ROOT / "notebooks" / "modelo_v3b_estructural_polaridad.ipynb"
)
CARPETA_OOF_BASE = (
    ROOT / "results" / "comparacion_ensamble_20261005_213910"
)

assert (DATA_DIR / "train.csv").exists(), "No se encuentra train.csv."
assert NOTEBOOK_ORIGINAL.exists(), "No se encuentra el notebook v3b."
assert CARPETA_OOF_BASE.exists(), "No se encuentran las predicciones OOF."

train_df = pd.read_csv(DATA_DIR / "train.csv")

assert {"id", "text", "label"}.issubset(train_df.columns)
assert train_df[["id", "text", "label"]].notna().all().all()
assert train_df["id"].is_unique
assert train_df["text"].str.strip().ne("").all()
assert set(train_df["label"]) == set(CLASES)

X = train_df["text"].to_numpy()
y = train_df["label"].to_numpy()

print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)

print("\nProyecto:", ROOT)
print("Dimensiones:", train_df.shape)
print("\nReseñas por clase:")
print(train_df["label"].value_counts())

print(
    "\nArchivos OOF disponibles:",
    len(list(CARPETA_OOF_BASE.glob("*.npz"))),
)

Python: 3.12.4
scikit-learn: 1.4.2
NumPy: 1.26.4
pandas: 2.2.2

Proyecto: /Users/alejandrobenavidesrubio/MLP1
Dimensiones: (12000, 3)

Reseñas por clase:
label
positivo    4212
negativo    4212
neutral     3576
Name: count, dtype: int64

Archivos OOF disponibles: 50


## 2. Extracción de opiniones contrapuestas

Reutilizamos el inventario y las reglas del notebook v3b, sin cargar
modelos entrenados ni ajustar un corrector ortográfico.

La primera versión utiliza texto normalizado y polaridad semántica fija.
Una reseña será candidata cuando se detecten exactamente dos opiniones
de signos contrarios.

La selección de candidatos depende exclusivamente del texto. Las
etiquetas se utilizarán posteriormente para entrenar y evaluar al
especialista dentro de cada fold.

Las reglas proceden de análisis anteriores del dataset; por ello,
la evaluación sigue siendo exploratoria.

In [2]:
import ast
import json
import re
import unicodedata

with NOTEBOOK_ORIGINAL.open(encoding="utf-8") as archivo:
    notebook_fuente = json.load(archivo)

# Definiciones revisadas: normalización, inventario y parser.
for numero_celda in (11, 34, 35):
    codigo = "".join(
        notebook_fuente["cells"][numero_celda - 1]["source"]
    )
    estructura = ast.parse(codigo)
    seleccion = []

    for nodo in estructura.body:
        if isinstance(nodo, ast.FunctionDef):
            # No necesitamos la función del corrector ortográfico.
            if nodo.name != "_ediciones1":
                seleccion.append(nodo)

        elif isinstance(nodo, ast.Assign):
            nombres = [
                destino.id
                for destino in nodo.targets
                if isinstance(destino, ast.Name)
            ]

            # Excluir el ejemplo y el alfabeto del corrector.
            if nombres and all(
                nombre not in {"demo", "LETRAS"}
                for nombre in nombres
            ):
                seleccion.append(nodo)

    modulo = ast.Module(body=seleccion, type_ignores=[])
    ast.fix_missing_locations(modulo)

    exec(
        compile(
            modulo,
            filename=f"{NOTEBOOK_ORIGINAL.name}:celda_{numero_celda}",
            mode="exec",
        ),
        globals(),
    )

print("Reglas del parser cargadas.")


def extraer_par_opiniones(texto):
    analisis = analizar_resena(normalizar(texto))
    opiniones = analisis["opiniones"]

    if len(opiniones) != 2:
        return None

    primera, ultima = opiniones
    signo_primera = polaridad_semantica(primera["clave"])
    signo_ultima = polaridad_semantica(ultima["clave"])

    if signo_primera == signo_ultima:
        return None

    return {
        "primera": primera,
        "ultima": ultima,
        "signo_primera": signo_primera,
        "signo_ultima": signo_ultima,
        "analisis": analisis,
    }


ejemplo = (
    "La batería me encantó. "
    "Pero la cámara no vale lo que cuesta."
)

par_demo = extraer_par_opiniones(ejemplo)
assert par_demo is not None

for posicion in ("primera", "ultima"):
    opinion = par_demo[posicion]

    print(f"\nOpinión {posicion}:")
    print("Expresión:", opinion["nucleo"])
    print("Clave:", opinion["clave"])
    print("Conector:", opinion["conector"] or "(ninguno)")
    print("Oración:", opinion["oracion_texto"])

Reglas del parser cargadas.

Opinión primera:
Expresión: me encanto
Clave: p_me_encanto
Conector: (ninguno)
Oración: la bateria me encanto.

Opinión ultima:
Expresión: no vale lo que cuesta
Clave: n_no_vale
Conector: pero
Oración: pero la camara no vale lo que cuesta.


## 3. Cobertura y objetivo del especialista

Identificamos candidatos mediante el texto: exactamente dos opiniones
detectadas, con polaridades semánticas opuestas.

Para las reseñas candidatas positivas o negativas, construimos el
objetivo del especialista: elegir la primera o la última opinión.

Las candidatas neutrales no tienen una respuesta válida para ese objetivo
binario. Se contabilizan aparte y permanecerán en la evaluación global.

Este paso es descriptivo. No ajusta vectorizadores ni modelos.

In [3]:
filas_candidatos = []

for indice, texto in enumerate(X):
    par = extraer_par_opiniones(texto)

    if par is None:
        continue

    etiqueta_primera = (
        "positivo" if par["signo_primera"] == 1 else "negativo"
    )
    etiqueta_ultima = (
        "positivo" if par["signo_ultima"] == 1 else "negativo"
    )

    etiqueta_real = y[indice]

    if etiqueta_real == etiqueta_primera:
        objetivo = "primera"
    elif etiqueta_real == etiqueta_ultima:
        objetivo = "ultima"
    else:
        objetivo = "fuera_objetivo"

    primera = par["primera"]
    ultima = par["ultima"]

    filas_candidatos.append({
        "indice": indice,
        "id": train_df.iloc[indice]["id"],
        "text": texto,
        "label": etiqueta_real,
        "objetivo": objetivo,
        "etiqueta_primera": etiqueta_primera,
        "etiqueta_ultima": etiqueta_ultima,
        "clave_primera": primera["clave"],
        "clave_ultima": ultima["clave"],
        "oracion_primera": primera["oracion_texto"],
        "oracion_ultima": ultima["oracion_texto"],
        "conector_ultima": ultima["conector"] or "(ninguno)",
        "ambas_frases_fijas": (
            not es_adjetival(primera)
            and not es_adjetival(ultima)
        ),
    })

candidatos_df = pd.DataFrame(filas_candidatos)

assert not candidatos_df.empty, "No se detectaron candidatos."
assert candidatos_df["id"].is_unique

print("Total de reseñas:", len(train_df))
print("Candidatas:", len(candidatos_df))
print(
    "Cobertura:",
    f"{100 * len(candidatos_df) / len(train_df):.2f}%",
)

print("\nObjetivo del especialista:")
print(candidatos_df["objetivo"].value_counts())

print("\nDistribución según el tipo de opiniones:")
display(pd.crosstab(
    candidatos_df["ambas_frases_fijas"],
    candidatos_df["objetivo"],
).rename_axis(
    index="ambas opiniones son frases hechas"
))

print("\nEjemplos:")
display(
    candidatos_df[[
        "id",
        "label",
        "objetivo",
        "clave_primera",
        "clave_ultima",
        "conector_ultima",
    ]].head(8)
)

Total de reseñas: 12000
Candidatas: 5870
Cobertura: 48.92%

Objetivo del especialista:
objetivo
ultima     4931
primera     939
Name: count, dtype: int64

Distribución según el tipo de opiniones:


objetivo,primera,ultima
ambas opiniones son frases hechas,,
False,3,3939
True,936,992



Ejemplos:


,id,label,objetivo,clave_primera,clave_ultima,conector_ultima
0,1,positivo,ultima,a-:incumplidor,a+:decente,al final de cuentas
1,2,negativo,ultima,a+:durable,a-:desagradable,aun asi
2,4,positivo,ultima,a-:terrible,a+:decente,(ninguno)
3,6,positivo,ultima,a-:incomod,a+:funcional,y
4,7,negativo,primera,n_fue_error,p_vale_pena,y
5,8,positivo,ultima,a-:aparatos,a+:confiable,al final
6,9,negativo,ultima,a+:eficiente,a-:incumplidor,(ninguno)
7,10,positivo,ultima,a-:decepcionante,a+:comod,con todo


## 4. Especialista en dos frases hechas contrapuestas

El análisis descriptivo identifica 1.928 reseñas con exactamente dos
opiniones de signo contrario, ambas expresadas mediante frases hechas.

El especialista se aplicará únicamente a ese grupo, identificado mediante
reglas sobre el texto. En las demás reseñas se conservará el ensamble base.

Representamos por separado el contexto de la primera opinión, el de la
última y el texto completo mediante TF-IDF. Añadimos el conector como
variable categórica.

Utilizamos regresión logística para predecir primera o ultima.
Los vectorizadores y el clasificador se ajustarán dentro de cada fold.

El porcentaje de la clase mayoritaria es una referencia descriptiva,
no un resultado de validación.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.base import clone

# Selección basada únicamente en características del texto.
especialista_df = candidatos_df.loc[
    candidatos_df["ambas_frases_fijas"]
].copy()

# Conservar el índice original para usar los mismos folds del ensamble.
especialista_df = especialista_df.set_index(
    "indice",
    verify_integrity=True,
)

especialista_df["texto_completo"] = (
    especialista_df["text"].map(normalizar)
)

COLUMNAS_ESPECIALISTA = [
    "oracion_primera",
    "oracion_ultima",
    "texto_completo",
    "conector_ultima",
]


def tfidf_especialista():
    return TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=True,
        max_df=1.0,
    )


def crear_especialista():
    representacion = ColumnTransformer([
        (
            "primera",
            tfidf_especialista(),
            "oracion_primera",
        ),
        (
            "ultima",
            tfidf_especialista(),
            "oracion_ultima",
        ),
        (
            "completo",
            tfidf_especialista(),
            "texto_completo",
        ),
        (
            "conector",
            OneHotEncoder(handle_unknown="ignore"),
            ["conector_ultima"],
        ),
    ])

    return Pipeline([
        ("representacion", representacion),
        (
            "clasificador",
            LogisticRegression(
                C=1.0,
                max_iter=5000,
                solver="lbfgs",
            ),
        ),
    ])


assert especialista_df.index.is_unique
assert especialista_df[COLUMNAS_ESPECIALISTA].notna().all().all()

print("Reseñas candidatas al especialista:", len(especialista_df))
print("\nObjetivos:")
print(especialista_df["objetivo"].value_counts())

proporciones = especialista_df["objetivo"].value_counts(
    normalize=True
)
print(
    "\nProporción de la clase mayoritaria:",
    f"{proporciones.max():.2%}",
)

clone(crear_especialista())
print("\nEspecialista preparado; todavía no entrenado.")

Reseñas candidatas al especialista: 1928

Objetivos:
objetivo
ultima     992
primera    936
Name: count, dtype: int64

Proporción de la clase mayoritaria: 51.45%

Especialista preparado; todavía no entrenado.


## 5. Evaluación del especialista dentro del ensamble

Utilizamos las mismas particiones estratificadas del ensamble base:
cinco folds y semillas 42 y 2024.

En cada fold, el especialista se ajusta exclusivamente con las reseñas
candidatas del entrenamiento. Después predice primera o ultima en
las candidatas de validación.

Comparamos:

- Ensamble original.
- Ensamble con sustitución por el especialista en las candidatas.
- Referencia que siempre elige la última opinión en las candidatas.

La sustitución depende únicamente del texto. Las etiquetas de validación
se utilizan solo para medir los resultados.

No se ajustan umbrales ni pesos en esta prueba.

In [5]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import time

PESOS_BASE = {
    "stacking": 0.20,
    "lr_palabras": 0.30,
    "estructural_original": 0.30,
    "est_lr": 0.20,
}

resultados_especialista = []
predicciones_experimento = {}

inicio = time.perf_counter()

for semilla in SEMILLAS:
    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=semilla,
    )
    folds = list(cv.split(X, y))

    # Reconstruir el ensamble y comprobar que los folds coinciden.
    proba_base = np.zeros((len(y), len(CLASES)))

    for nombre, peso in PESOS_BASE.items():
        proba_componente = np.full((len(y), len(CLASES)), np.nan)

        for fold, (_, va_idx) in enumerate(folds, start=1):
            ruta = CARPETA_OOF_BASE / (
                f"semilla_{semilla}_fold_{fold}_{nombre}.npz"
            )
            assert ruta.exists(), f"Falta: {ruta.name}"

            with np.load(ruta, allow_pickle=False) as archivo:
                assert np.array_equal(archivo["indices"], va_idx)
                assert np.array_equal(
                    archivo["ids"],
                    train_df["id"].to_numpy()[va_idx],
                )
                assert np.array_equal(archivo["clases"], CLASES)

                proba_componente[va_idx] = archivo["probabilidades"]

        assert np.isfinite(proba_componente).all()
        proba_base += peso * proba_componente

    pred_base = CLASES[proba_base.argmax(axis=1)]
    pred_nueva = pred_base.copy()
    pred_siempre_ultima = pred_base.copy()

    cobertura = np.zeros(len(y), dtype=int)

    print(f"\n=== Semilla {semilla} ===", flush=True)

    for fold, (tr_idx, va_idx) in enumerate(folds, start=1):
        entrenamiento = especialista_df.loc[
            especialista_df.index.isin(tr_idx)
            & especialista_df["objetivo"].isin(["primera", "ultima"])
        ]

        # No filtrar validación por su etiqueta real.
        validacion = especialista_df.loc[
            especialista_df.index.isin(va_idx)
        ]

        assert set(entrenamiento.index).isdisjoint(validacion.index)
        assert entrenamiento["objetivo"].nunique() == 2

        modelo = crear_especialista()
        modelo.fit(
            entrenamiento[COLUMNAS_ESPECIALISTA],
            entrenamiento["objetivo"],
        )

        eleccion = modelo.predict(
            validacion[COLUMNAS_ESPECIALISTA]
        )

        pred_sentimiento = np.where(
            eleccion == "primera",
            validacion["etiqueta_primera"].to_numpy(),
            validacion["etiqueta_ultima"].to_numpy(),
        )

        indices = validacion.index.to_numpy()

        pred_nueva[indices] = pred_sentimiento
        pred_siempre_ultima[indices] = (
            validacion["etiqueta_ultima"].to_numpy()
        )
        cobertura[indices] += 1

        acc_base_grupo = accuracy_score(y[indices], pred_base[indices])
        acc_nueva_grupo = accuracy_score(y[indices], pred_nueva[indices])

        print(
            f"Fold {fold}/5 | candidatas: {len(indices)}"
            f" | base: {acc_base_grupo:.4f}"
            f" | especialista: {acc_nueva_grupo:.4f}",
            flush=True,
        )

    indices_grupo = especialista_df.index.to_numpy()
    assert (cobertura[indices_grupo] == 1).all()

    mascara_grupo = np.zeros(len(y), dtype=bool)
    mascara_grupo[indices_grupo] = True
    assert np.array_equal(
        pred_nueva[~mascara_grupo],
        pred_base[~mascara_grupo],
    )

    correcto_base = pred_base == y
    correcto_nuevo = pred_nueva == y

    resultados_especialista.append({
        "semilla": semilla,
        "base_grupo": accuracy_score(
            y[indices_grupo], pred_base[indices_grupo]
        ),
        "especialista_grupo": accuracy_score(
            y[indices_grupo], pred_nueva[indices_grupo]
        ),
        "siempre_ultima_grupo": accuracy_score(
            y[indices_grupo], pred_siempre_ultima[indices_grupo]
        ),
        "base_global": correcto_base.mean(),
        "nuevo_global": correcto_nuevo.mean(),
        "mejora_pp": 100 * (
            correcto_nuevo.mean() - correcto_base.mean()
        ),
        "corregidos": int((~correcto_base & correcto_nuevo).sum()),
        "introducidos": int((correcto_base & ~correcto_nuevo).sum()),
    })

    predicciones_experimento[semilla] = {
        "base": pred_base,
        "especialista": pred_nueva,
    }

resumen_especialista = pd.DataFrame(
    resultados_especialista
).set_index("semilla")

display(resumen_especialista.round(6))

print(
    "Tiempo total:",
    round((time.perf_counter() - inicio) / 60, 1),
    "minutos",
)


=== Semilla 42 ===
Fold 1/5 | candidatas: 398 | base: 0.5402 | especialista: 0.5025
Fold 2/5 | candidatas: 386 | base: 0.4974 | especialista: 0.5492
Fold 3/5 | candidatas: 382 | base: 0.5576 | especialista: 0.5131
Fold 4/5 | candidatas: 366 | base: 0.5574 | especialista: 0.5519
Fold 5/5 | candidatas: 396 | base: 0.5328 | especialista: 0.4924

=== Semilla 2024 ===
Fold 1/5 | candidatas: 380 | base: 0.5395 | especialista: 0.5079
Fold 2/5 | candidatas: 369 | base: 0.5420 | especialista: 0.5718
Fold 3/5 | candidatas: 408 | base: 0.5196 | especialista: 0.4951
Fold 4/5 | candidatas: 398 | base: 0.5402 | especialista: 0.5201
Fold 5/5 | candidatas: 373 | base: 0.5308 | especialista: 0.4960


,base_grupo,especialista_grupo,siempre_ultima_grupo,base_global,nuevo_global,mejora_pp,corregidos,introducidos
semilla,,,,,,,,
42,0.536826,0.521266,0.514523,0.916500,0.914000,-0.250000,411,441
2024,0.534232,0.517635,0.514523,0.916583,0.913917,-0.266667,401,433


Tiempo total: 0.0 minutos


### Conclusión del especialista

El especialista obtuvo accuracies de 0.521266 y 0.517635 en las
reseñas candidatas, inferiores a los del ensamble base.

Su incorporación redujo el accuracy global:

- Semilla 42: 0.916500 → 0.914000.
- Semilla 2024: 0.916583 → 0.913917.

Aunque corrigió errores, introdujo más de los que corrigió.
No se adopta esta sustitución ni se genera un submission.

El experimento no demuestra que las etiquetas sean aleatorias;
únicamente descarta esta configuración como mejora del ensamble.

## 6. Diagnóstico de reproducción de v3b

Comparamos los resultados por componente de nuestra ejecución con
los registrados en el notebook original, manteniendo el entorno base.

Las ejecuciones utilizan versiones distintas de las librerías.
Esta comparación permite localizar diferencias, pero no determinar
por sí sola su causa.

Reutilizamos las probabilidades OOF guardadas, sin reentrenar modelos
ni modificar el entorno.

In [6]:
# Valores redondeados registrados en el notebook original.
referencias_v3b = {
    42: {
        "stacking": 0.9012,
        "lr_palabras": 0.8951,
        "estructural_original": 0.9075,
        "est_lr": 0.9126,
    },
    2024: {
        "stacking": 0.9004,
        "lr_palabras": 0.8939,
        "estructural_original": 0.9096,
        "est_lr": 0.9116,
    },
}

filas_reproduccion = []

for semilla, referencias in referencias_v3b.items():
    for componente, referencia in referencias.items():
        probabilidades = np.full((len(y), len(CLASES)), np.nan)
        cobertura = np.zeros(len(y), dtype=int)

        archivos = sorted(CARPETA_OOF_BASE.glob(
            f"semilla_{semilla}_fold_*_{componente}.npz"
        ))
        assert len(archivos) == 5

        for ruta in archivos:
            with np.load(ruta, allow_pickle=False) as archivo:
                indices = archivo["indices"]

                assert np.array_equal(
                    archivo["ids"],
                    train_df["id"].to_numpy()[indices],
                )
                assert np.array_equal(archivo["clases"], CLASES)

                probabilidades[indices] = archivo["probabilidades"]
                cobertura[indices] += 1

        assert (cobertura == 1).all()
        assert np.isfinite(probabilidades).all()

        predicciones = CLASES[probabilidades.argmax(axis=1)]
        accuracy_actual = np.mean(predicciones == y)

        filas_reproduccion.append({
            "semilla": semilla,
            "componente": componente,
            "referencia_redondeada": referencia,
            "accuracy_actual": accuracy_actual,
            "diferencia_pp_aprox": 100 * (
                accuracy_actual - referencia
            ),
        })

diagnostico_reproduccion = pd.DataFrame(
    filas_reproduccion
).set_index(["semilla", "componente"])

display(diagnostico_reproduccion.round(6))

referencia_redondeada  accuracy_actual  \
semilla componente                                                     
42      stacking                             0.9012         0.900417   
        lr_palabras                          0.8951         0.895083   
        estructural_original                 0.9075         0.907500   
        est_lr                               0.9126         0.912917   
2024    stacking                             0.9004         0.900750   
        lr_palabras                          0.8939         0.893917   
        estructural_original                 0.9096         0.909583   
        est_lr                               0.9116         0.911417   

                              diferencia_pp_aprox  
semilla componente                                 
42      stacking                        -0.078333  
        lr_palabras                     -0.001667  
        estructural_original             0.000000  
        est_lr                           0.031667  
2024    stacking                         0.035000  
        lr_palabras                      0.001667  
        estructural_original            -0.001667  
        est_lr                          -0.018333

### Resultado del diagnóstico de reproducción

LR por bloques y el componente estructural coinciden con los resultados
originales a la precisión registrada.

Stacking y est_lr presentan diferencias pequeñas cuya dirección cambia
entre semillas. No se identifica un componente sistemáticamente peor.

Las versiones del entorno son distintas, pero esta comparación no
demuestra que sean la causa de la discrepancia. Para comparar las
predicciones y probabilidades exactamente se necesitan las OOF originales.

Se conserva el entorno base y no se repiten entrenamientos únicamente
para intentar igualar una cifra registrada.

## 7. Combinación suave del ensamble y el especialista

En lugar de sustituir completamente al ensamble, evaluamos una mezcla
fija de probabilidades en las reseñas candidatas:

- Ensamble base: 90%.
- Especialista: 10%.

En las demás reseñas conservamos el ensamble base.

El especialista predice primera o ultima. Convertimos esas probabilidades
a negativo, neutral y positivo según la polaridad de cada opinión.
El especialista binario asigna probabilidad cero a neutral; el ensamble
conserva su aporte a esa clase.

Utilizamos probabilidades fuera de muestra de ambos modelos y las mismas
particiones. El peso se fija antes de evaluar esta combinación y no se
selecciona mediante una búsqueda sobre estos resultados.

Esta evaluación sigue siendo exploratoria.

In [7]:
from datetime import datetime
from sklearn.model_selection import StratifiedKFold
import time

PESO_ESPECIALISTA = 0.10

PESOS_BASE_MEZCLA = {
    "stacking": 0.20,
    "lr_palabras": 0.30,
    "estructural_original": 0.30,
    "est_lr": 0.20,
}

marca_mezcla = datetime.now().strftime("%Y%m%d_%H%M%S")
carpeta_mezcla = (
    ROOT / "results" / f"mezcla_especialista_{marca_mezcla}"
)
carpeta_mezcla.mkdir(parents=True, exist_ok=False)

resultados_mezcla = []
oof_mezcla = {}
inicio_mezcla = time.perf_counter()

for semilla in SEMILLAS:
    folds = list(
        StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=semilla,
        ).split(X, y)
    )

    # 1. Recuperar las probabilidades OOF del ensamble.
    proba_base = np.zeros((len(y), len(CLASES)))

    for nombre, peso in PESOS_BASE_MEZCLA.items():
        proba_componente = np.full((len(y), len(CLASES)), np.nan)

        for fold, (_, va_idx) in enumerate(folds, start=1):
            ruta = CARPETA_OOF_BASE / (
                f"semilla_{semilla}_fold_{fold}_{nombre}.npz"
            )

            with np.load(ruta, allow_pickle=False) as archivo:
                assert np.array_equal(archivo["indices"], va_idx)
                assert np.array_equal(
                    archivo["ids"],
                    train_df["id"].to_numpy()[va_idx],
                )
                assert np.array_equal(archivo["clases"], CLASES)

                proba_componente[va_idx] = archivo["probabilidades"]

        assert np.isfinite(proba_componente).all()
        assert np.allclose(proba_componente.sum(axis=1), 1)
        proba_base += peso * proba_componente

    # Solo rellenamos probabilidades del especialista en candidatas.
    proba_especialista = np.full((len(y), len(CLASES)), np.nan)
    cobertura = np.zeros(len(y), dtype=int)

    print(f"\n=== Semilla {semilla} ===", flush=True)

    # 2. Obtener probabilidades OOF del especialista.
    for fold, (tr_idx, va_idx) in enumerate(folds, start=1):
        entrenamiento = especialista_df.loc[
            especialista_df.index.isin(tr_idx)
            & especialista_df["objetivo"].isin(["primera", "ultima"])
        ]
        validacion = especialista_df.loc[
            especialista_df.index.isin(va_idx)
        ]

        assert set(entrenamiento.index).isdisjoint(validacion.index)
        assert entrenamiento["objetivo"].nunique() == 2

        candidato = crear_especialista()
        candidato.fit(
            entrenamiento[COLUMNAS_ESPECIALISTA],
            entrenamiento["objetivo"],
        )

        indices = validacion.index.to_numpy()

        if len(indices) == 0:
            continue

        probabilidades_posicion = candidato.predict_proba(
            validacion[COLUMNAS_ESPECIALISTA]
        )

        # Convertir primera/ultima a negativo/neutral/positivo.
        probabilidades_sentimiento = np.zeros((len(indices), len(CLASES)))
        filas = np.arange(len(indices))

        for posicion in ("primera", "ultima"):
            columna_posicion = list(candidato.classes_).index(posicion)
            etiquetas = validacion[f"etiqueta_{posicion}"].to_numpy()

            columnas_sentimiento = np.array([
                list(CLASES).index(etiqueta)
                for etiqueta in etiquetas
            ])

            probabilidades_sentimiento[
                filas, columnas_sentimiento
            ] += probabilidades_posicion[:, columna_posicion]

        assert np.allclose(
            probabilidades_sentimiento.sum(axis=1), 1
        )

        proba_especialista[indices] = probabilidades_sentimiento
        cobertura[indices] += 1

        print(
            f"Fold {fold}/5 terminado: {len(indices)} candidatas.",
            flush=True,
        )

    indices_grupo = especialista_df.index.to_numpy()
    assert (cobertura[indices_grupo] == 1).all()
    assert np.isfinite(proba_especialista[indices_grupo]).all()

    # 3. Mezclar exclusivamente en las candidatas.
    proba_combinada = proba_base.copy()
    proba_combinada[indices_grupo] = (
        (1 - PESO_ESPECIALISTA) * proba_base[indices_grupo]
        + PESO_ESPECIALISTA * proba_especialista[indices_grupo]
    )

    mascara_grupo = np.zeros(len(y), dtype=bool)
    mascara_grupo[indices_grupo] = True

    assert np.array_equal(
        proba_combinada[~mascara_grupo],
        proba_base[~mascara_grupo],
    )
    assert np.allclose(proba_combinada.sum(axis=1), 1)

    pred_base = CLASES[proba_base.argmax(axis=1)]
    pred_mezcla = CLASES[proba_combinada.argmax(axis=1)]

    correcto_base = pred_base == y
    correcto_mezcla = pred_mezcla == y

    resultados_mezcla.append({
        "semilla": semilla,
        "base_grupo": correcto_base[indices_grupo].mean(),
        "mezcla_grupo": correcto_mezcla[indices_grupo].mean(),
        "base_global": correcto_base.mean(),
        "mezcla_global": correcto_mezcla.mean(),
        "mejora_pp": 100 * (
            correcto_mezcla.mean() - correcto_base.mean()
        ),
        "corregidos": int((~correcto_base & correcto_mezcla).sum()),
        "introducidos": int((correcto_base & ~correcto_mezcla).sum()),
        "predicciones_cambiadas": int((pred_base != pred_mezcla).sum()),
    })

    oof_mezcla[semilla] = {
        "base": proba_base,
        "especialista_candidatas": proba_especialista[indices_grupo],
        "indices_candidatas": indices_grupo,
        "mezcla": proba_combinada,
    }

    np.savez_compressed(
        carpeta_mezcla / f"oof_semilla_{semilla}.npz",
        ids=train_df["id"].to_numpy(),
        clases=CLASES,
        indices_candidatas=indices_grupo,
        base=proba_base,
        especialista_candidatas=proba_especialista[indices_grupo],
        mezcla=proba_combinada,
        peso_especialista=PESO_ESPECIALISTA,
    )

resumen_mezcla = pd.DataFrame(resultados_mezcla).set_index("semilla")
resumen_mezcla.to_csv(carpeta_mezcla / "resultados.csv")

display(resumen_mezcla.round(6))

print(
    "Tiempo total:",
    round((time.perf_counter() - inicio_mezcla) / 60, 1),
    "minutos",
)
print("Resultados guardados en:", carpeta_mezcla)


=== Semilla 42 ===
Fold 1/5 terminado: 398 candidatas.
Fold 2/5 terminado: 386 candidatas.
Fold 3/5 terminado: 382 candidatas.
Fold 4/5 terminado: 366 candidatas.
Fold 5/5 terminado: 396 candidatas.

=== Semilla 2024 ===
Fold 1/5 terminado: 380 candidatas.
Fold 2/5 terminado: 369 candidatas.
Fold 3/5 terminado: 408 candidatas.
Fold 4/5 terminado: 398 candidatas.
Fold 5/5 terminado: 373 candidatas.


,base_grupo,mezcla_grupo,base_global,mezcla_global,mejora_pp,corregidos,introducidos,predicciones_cambiadas
semilla,,,,,,,,
42,0.536826,0.534232,0.916500,0.916083,-0.041667,40,45,85
2024,0.534232,0.536307,0.916583,0.916917,0.033333,39,35,74


Tiempo total: 0.0 minutos
Resultados guardados en: /Users/alejandrobenavidesrubio/MLP1/results/mezcla_especialista_20261006_141215


Se evaluaron mezclas con participaciones del especialista del 5%, 10%
y 20%, además de la sustitución completa.

Ninguna mejoró consistentemente al ensamble base en ambas semillas.
Se conserva el ensamble original. No se continúa ajustando el peso
sobre estas mismas particiones para evitar seleccionar fluctuaciones
de la validación.

## 8. Promedio de versiones de est_lr

### Hipótesis

El componente est_lr utiliza particiones internas para construir las
características con las que entrena su clasificador final. Promediar
versiones con distintas particiones podría reducir esa variación.

### Diseño

En cada fold externo se combinan tres versiones, con semillas internas
1000 + k, 2000 + k y 3000 + k, donde k comienza en cero.

Reutilizamos las probabilidades OOF guardadas de la primera versión.
Las otras dos se entrenan únicamente con el entrenamiento del fold.

Promediamos sus probabilidades con pesos iguales y conservamos el
peso total de est_lr en el ensamble: 0.20.

Primero evaluamos la partición externa con semilla 42. Si mejora,
comprobamos la segunda partición. No seleccionamos semillas según
sus resultados individuales.

La evaluación sigue siendo exploratoria.

In [10]:
from scipy.sparse import hstack

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from threadpoolctl import threadpool_limits

threadpool_limits(limits=2)

# Leer nuevamente el notebook fuente.
with NOTEBOOK_ORIGINAL.open(encoding="utf-8") as archivo:
    fuente_est_lr = json.load(archivo)


def cargar_definiciones_est_lr(numero_celda):
    celda = fuente_est_lr["cells"][numero_celda - 1]
    assert celda["cell_type"] == "code"

    codigo = "".join(celda["source"])

    if numero_celda == 48:
        marcador = "# Ejemplo de segmentación"
        assert marcador in codigo, (
            "La celda original cambió; revisar antes de continuar."
        )
        codigo = codigo.split(marcador, maxsplit=1)[0]

    estructura = ast.parse(codigo)
    nodos = []

    for nodo in estructura.body:
        if isinstance(
            nodo,
            (ast.FunctionDef, ast.ClassDef, ast.Assign),
        ):
            nodos.append(nodo)

        elif numero_celda == 48 and isinstance(nodo, ast.For):
            # Bucles que construyen el diccionario de conectores.
            assert (
                isinstance(nodo.target, ast.Name)
                and nodo.target.id == "_c"
            )
            nodos.append(nodo)

        elif isinstance(nodo, ast.Expr):
            # Omitir impresiones y demostraciones.
            continue

        else:
            raise ValueError(
                f"Instrucción inesperada en celda {numero_celda}: "
                f"{type(nodo).__name__}"
            )

    modulo = ast.Module(body=nodos, type_ignores=[])
    ast.fix_missing_locations(modulo)

    exec(
        compile(
            modulo,
            filename=f"{NOTEBOOK_ORIGINAL.name}:celda_{numero_celda}",
            mode="exec",
        ),
        globals(),
    )

    print(f"Definiciones de celda {numero_celda}: cargadas.")


for numero_celda in (48, 49, 50):
    cargar_definiciones_est_lr(numero_celda)

SEMILLAS_INTERNAS_PROMEDIO = (1000, 2000, 3000)

for semilla_interna in SEMILLAS_INTERNAS_PROMEDIO:
    modelo_prueba = clone(
        modelo_est_lr(semilla_interna=semilla_interna)
    )
    print(modelo_prueba.get_params())

print("\nPreparación completa. No se han entrenado modelos.")

Definiciones de celda 48: cargadas.
Definiciones de celda 49: cargadas.
Definiciones de celda 50: cargadas.
{'C': 1.0, 'n_folds_internos': 5, 'semilla_interna': 1000}
{'C': 1.0, 'n_folds_internos': 5, 'semilla_interna': 2000}
{'C': 1.0, 'n_folds_internos': 5, 'semilla_interna': 3000}

Preparación completa. No se han entrenado modelos.


In [11]:
from datetime import datetime
from sklearn.preprocessing import LabelEncoder
import time

# Mantener la misma carpeta si se vuelve a ejecutar esta celda.
if "carpeta_promedio_est_lr" not in globals():
    marca_promedio = datetime.now().strftime("%Y%m%d_%H%M%S")
    carpeta_promedio_est_lr = (
        ROOT / "results" / f"promedio_est_lr_{marca_promedio}"
    )
    carpeta_promedio_est_lr.mkdir(parents=True, exist_ok=False)

SEMILLA_EXTERNA_PROMEDIO = 42

folds_promedio = list(
    StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEMILLA_EXTERNA_PROMEDIO,
    ).split(X, y)
)

codificador_promedio = LabelEncoder().fit(CLASES)
y_codificado_promedio = codificador_promedio.transform(y)
assert np.array_equal(codificador_promedio.classes_, CLASES)

pesos_promedio = {
    "stacking": 0.20,
    "lr_palabras": 0.30,
    "estructural_original": 0.30,
    "est_lr": 0.20,
}

# 1. Recuperar probabilidades originales.
oof_base_promedio = {}

for nombre in pesos_promedio:
    matriz = np.full((len(y), len(CLASES)), np.nan)

    for fold, (_, va_idx) in enumerate(folds_promedio, start=1):
        ruta = CARPETA_OOF_BASE / (
            f"semilla_42_fold_{fold}_{nombre}.npz"
        )

        with np.load(ruta, allow_pickle=False) as archivo:
            assert np.array_equal(archivo["indices"], va_idx)
            assert np.array_equal(
                archivo["ids"],
                train_df["id"].to_numpy()[va_idx],
            )
            assert np.array_equal(archivo["clases"], CLASES)

            matriz[va_idx] = archivo["probabilidades"]

    assert np.isfinite(matriz).all()
    assert np.allclose(matriz.sum(axis=1), 1)
    oof_base_promedio[nombre] = matriz

# La primera versión ya se calculó con semillas internas 1000 + k.
oof_versiones_est_lr = {
    1000: oof_base_promedio["est_lr"],
}

inicio_promedio = time.perf_counter()

# 2. Entrenar las dos versiones adicionales.
for semilla_inicial in (2000, 3000):
    matriz = np.full((len(y), len(CLASES)), np.nan)
    cobertura = np.zeros(len(y), dtype=int)

    print(
        f"\n=== Semilla interna: {semilla_inicial} + k ===",
        flush=True,
    )

    for k, (tr_idx, va_idx) in enumerate(folds_promedio):
        ruta = carpeta_promedio_est_lr / (
            f"externa_42_interna_{semilla_inicial}_fold_{k + 1}.npz"
        )

        if ruta.exists():
            with np.load(ruta, allow_pickle=False) as archivo:
                assert np.array_equal(archivo["indices"], va_idx)
                assert np.array_equal(
                    archivo["ids"],
                    train_df["id"].to_numpy()[va_idx],
                )
                assert np.array_equal(archivo["clases"], CLASES)
                probabilidades = archivo["probabilidades"]

            print(f"Fold {k + 1}/5 recuperado.", flush=True)

        else:
            inicio_fold = time.perf_counter()
            print(f"Entrenando fold {k + 1}/5...", flush=True)

            candidato = modelo_est_lr(
                semilla_interna=semilla_inicial + k
            )
            candidato.fit(
                X[tr_idx],
                y_codificado_promedio[tr_idx],
            )

            orden = [
                list(candidato.classes_).index(i)
                for i in range(len(CLASES))
            ]
            probabilidades = candidato.predict_proba(
                X[va_idx]
            )[:, orden]

            assert np.isfinite(probabilidades).all()
            assert np.allclose(probabilidades.sum(axis=1), 1)

            np.savez_compressed(
                ruta,
                indices=va_idx,
                ids=train_df["id"].to_numpy()[va_idx],
                clases=CLASES,
                probabilidades=probabilidades,
            )

            print(
                f"Fold {k + 1}/5 terminado en "
                f"{time.perf_counter() - inicio_fold:.1f} s.",
                flush=True,
            )
            del candidato

        assert probabilidades.shape == (len(va_idx), len(CLASES))
        assert np.isfinite(probabilidades).all()
        assert np.allclose(probabilidades.sum(axis=1), 1)

        matriz[va_idx] = probabilidades
        cobertura[va_idx] += 1

    assert (cobertura == 1).all()
    oof_versiones_est_lr[semilla_inicial] = matriz

# 3. Promediar las tres versiones con pesos iguales.
proba_est_lr_promediada = np.mean(
    [
        oof_versiones_est_lr[semilla]
        for semilla in SEMILLAS_INTERNAS_PROMEDIO
    ],
    axis=0,
)

proba_ensamble_original = sum(
    peso * oof_base_promedio[nombre]
    for nombre, peso in pesos_promedio.items()
)

proba_ensamble_promedio = (
    sum(
        pesos_promedio[nombre] * oof_base_promedio[nombre]
        for nombre in ("stacking", "lr_palabras", "estructural_original")
    )
    + pesos_promedio["est_lr"] * proba_est_lr_promediada
)

pred_original_promedio = CLASES[
    proba_ensamble_original.argmax(axis=1)
]
pred_nuevo_promedio = CLASES[
    proba_ensamble_promedio.argmax(axis=1)
]

aciertos_original = pred_original_promedio == y
aciertos_nuevo = pred_nuevo_promedio == y

resumen_promedio_est_lr = pd.DataFrame([{
    "semilla": SEMILLA_EXTERNA_PROMEDIO,
    "base_global": aciertos_original.mean(),
    "promedio_global": aciertos_nuevo.mean(),
    "mejora_pp": 100 * (
        aciertos_nuevo.mean() - aciertos_original.mean()
    ),
    "corregidos": int((~aciertos_original & aciertos_nuevo).sum()),
    "introducidos": int((aciertos_original & ~aciertos_nuevo).sum()),
    "predicciones_cambiadas": int(
        (pred_original_promedio != pred_nuevo_promedio).sum()
    ),
}]).set_index("semilla")

resumen_promedio_est_lr.to_csv(
    carpeta_promedio_est_lr / "resultados_semilla_42.csv"
)

display(resumen_promedio_est_lr.round(6))

print(
    "Tiempo total:",
    round((time.perf_counter() - inicio_promedio) / 60, 1),
    "minutos",
)
print("Resultados guardados en:", carpeta_promedio_est_lr)


=== Semilla interna: 2000 + k ===
Entrenando fold 1/5...
Fold 1/5 terminado en 70.3 s.
Entrenando fold 2/5...
Fold 2/5 terminado en 67.1 s.
Entrenando fold 3/5...
Fold 3/5 terminado en 68.8 s.
Entrenando fold 4/5...
Fold 4/5 terminado en 66.7 s.
Entrenando fold 5/5...
Fold 5/5 terminado en 67.8 s.

=== Semilla interna: 3000 + k ===
Entrenando fold 1/5...
Fold 1/5 terminado en 73.1 s.
Entrenando fold 2/5...
Fold 2/5 terminado en 73.6 s.
Entrenando fold 3/5...
Fold 3/5 terminado en 67.7 s.
Entrenando fold 4/5...
Fold 4/5 terminado en 67.8 s.
Entrenando fold 5/5...
Fold 5/5 terminado en 66.9 s.


,base_global,promedio_global,mejora_pp,corregidos,introducidos,predicciones_cambiadas
semilla,,,,,,
42,0.9165,0.915583,-0.091667,6,17,23


Tiempo total: 11.5 minutos
Resultados guardados en: /Users/alejandrobenavidesrubio/MLP1/results/promedio_est_lr_20261006_141850


### Conclusión del promedio de est_lr

Promediar tres versiones con distintas semillas internas redujo el
accuracy del ensamble de 0.916500 a 0.915583 en la partición externa 42.

La variante corrigió 6 errores e introdujo 17. Siguiendo el criterio
definido previamente, no se continúa con la segunda partición ni se
genera un submission.

Se conserva la configuración original de est_lr. Este resultado
descarta la combinación probada; no demuestra que cualquier promedio
de modelos sea perjudicial.

## 9. Experimento: SVM no lineal sobre características estructurales

### Hipótesis

Un SVM con kernel RBF puede aprovechar relaciones entre las características
estructurales de forma diferente a HistGradientBoosting y regresión logística.

### Configuración

- Las mismas 59 características estructurales.
- StandardScaler ajustado dentro de cada fold.
- SVC con kernel RBF, C=1 y gamma="scale".
- Probabilidades habilitadas para evaluar su incorporación al ensamble.
- Semilla fija para la estimación de probabilidades.

Se mantienen las particiones externas del experimento. No se utilizan
eval.csv ni resultados de Kaggle para elegir los parámetros.

In [12]:
import collections

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

with NOTEBOOK_ORIGINAL.open(encoding="utf-8") as archivo:
    fuente_svm = json.load(archivo)

# Recuperar las piezas del corrector que omitimos al preparar
# el especialista. Su vocabulario se aprenderá dentro de cada fold.
codigo_corrector = "".join(
    fuente_svm["cells"][33]["source"]
)
arbol_corrector = ast.parse(codigo_corrector)

nodos_corrector = []

for nodo in arbol_corrector.body:
    if isinstance(nodo, (ast.FunctionDef, ast.ClassDef)):
        if nodo.name in {"_ediciones1", "CorrectorTypos"}:
            nodos_corrector.append(nodo)

    elif isinstance(nodo, ast.Assign):
        if any(
            isinstance(destino, ast.Name)
            and destino.id == "LETRAS"
            for destino in nodo.targets
        ):
            nodos_corrector.append(nodo)

assert len(nodos_corrector) == 3

modulo_corrector = ast.Module(
    body=nodos_corrector,
    type_ignores=[],
)
ast.fix_missing_locations(modulo_corrector)
exec(compile(modulo_corrector, "corrector_v3b", "exec"), globals())

# Recuperar las características y el transformador originales.
codigo_rasgos = "".join(
    fuente_svm["cells"][43]["source"]
)
arbol_rasgos = ast.parse(codigo_rasgos)

nombres_rasgos = {
    "rasgos_plantilla",
    "_es_positivo",
    "_es_negativo",
    "RasgosPlantilla",
}

nodos_rasgos = []

for nodo in arbol_rasgos.body:
    if isinstance(nodo, (ast.FunctionDef, ast.ClassDef)):
        if nodo.name in nombres_rasgos:
            nodos_rasgos.append(nodo)

    elif isinstance(nodo, ast.Assign):
        if any(
            isinstance(destino, ast.Name)
            and destino.id == "CONECTORES_RASGO"
            for destino in nodo.targets
        ):
            nodos_rasgos.append(nodo)

assert len(nodos_rasgos) == 5

modulo_rasgos = ast.Module(body=nodos_rasgos, type_ignores=[])
ast.fix_missing_locations(modulo_rasgos)
exec(compile(modulo_rasgos, "rasgos_v3b", "exec"), globals())


def crear_svm_estructural():
    return Pipeline([
        (
            "rasgos",
            RasgosPlantilla(modo_polaridad="aprendida"),
        ),
        ("escalador", StandardScaler()),
        (
            "clasificador",
            SVC(
                kernel="rbf",
                C=1.0,
                gamma="scale",
                probability=True,
                random_state=42,
                cache_size=512,
            ),
        ),
    ])


clone(crear_svm_estructural())

assert len(
    rasgos_plantilla(
        analizar_resena(normalizar(
            "La batería es excelente, pero la cámara es mala."
        ))
    )
) == 59

print("SVM estructural preparado con 59 características.")
print("Todavía no se han entrenado modelos.")

SVM estructural preparado con 59 características.
Todavía no se han entrenado modelos.


In [13]:
from datetime import datetime
from sklearn.model_selection import StratifiedKFold
import time

SEMILLA_SVM = 42

folds_svm = list(
    StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEMILLA_SVM,
    ).split(X, y)
)

pesos_base_svm = {
    "stacking": 0.20,
    "lr_palabras": 0.30,
    "estructural_original": 0.30,
    "est_lr": 0.20,
}

marca_svm = datetime.now().strftime("%Y%m%d_%H%M%S")
carpeta_svm = ROOT / "results" / f"svm_estructural_{marca_svm}"
carpeta_svm.mkdir(parents=True, exist_ok=False)

# 1. Recuperar las probabilidades del ensamble original.
oof_base_svm = {}

for nombre in pesos_base_svm:
    matriz = np.full((len(y), len(CLASES)), np.nan)

    for fold, (_, va_idx) in enumerate(folds_svm, start=1):
        ruta = CARPETA_OOF_BASE / (
            f"semilla_{SEMILLA_SVM}_fold_{fold}_{nombre}.npz"
        )

        with np.load(ruta, allow_pickle=False) as archivo:
            assert np.array_equal(archivo["indices"], va_idx)
            assert np.array_equal(
                archivo["ids"],
                train_df["id"].to_numpy()[va_idx],
            )
            assert np.array_equal(archivo["clases"], CLASES)

            matriz[va_idx] = archivo["probabilidades"]

    assert np.isfinite(matriz).all()
    assert np.allclose(matriz.sum(axis=1), 1)
    oof_base_svm[nombre] = matriz

# 2. Entrenar el SVM exclusivamente con entrenamiento de cada fold.
oof_svm = np.full((len(y), len(CLASES)), np.nan)
cobertura_svm = np.zeros(len(y), dtype=int)
resultados_folds_svm = []

inicio_svm = time.perf_counter()

for fold, (tr_idx, va_idx) in enumerate(folds_svm, start=1):
    print(f"\nEntrenando SVM: fold {fold}/5...", flush=True)
    inicio_fold = time.perf_counter()

    candidato_svm = crear_svm_estructural()
    candidato_svm.fit(X[tr_idx], y[tr_idx])

    orden = [
        list(candidato_svm.classes_).index(clase)
        for clase in CLASES
    ]
    probabilidades = candidato_svm.predict_proba(
        X[va_idx]
    )[:, orden]

    assert np.isfinite(probabilidades).all()
    assert np.allclose(probabilidades.sum(axis=1), 1)

    oof_svm[va_idx] = probabilidades
    cobertura_svm[va_idx] += 1

    # Evaluar mediante argmax de probabilidades, que será la salida
    # utilizada al combinarlo con los otros componentes.
    pred_svm_fold = CLASES[probabilidades.argmax(axis=1)]
    accuracy_fold = np.mean(pred_svm_fold == y[va_idx])

    resultados_folds_svm.append({
        "semilla": SEMILLA_SVM,
        "fold": fold,
        "accuracy_svm_probabilidades": accuracy_fold,
    })

    np.savez_compressed(
        carpeta_svm / f"semilla_{SEMILLA_SVM}_fold_{fold}.npz",
        indices=va_idx,
        ids=train_df["id"].to_numpy()[va_idx],
        clases=CLASES,
        probabilidades=probabilidades,
    )

    print(
        f"Accuracy SVM: {accuracy_fold:.4f}"
        f" | Tiempo: {time.perf_counter() - inicio_fold:.1f} s",
        flush=True,
    )

assert (cobertura_svm == 1).all()
assert np.isfinite(oof_svm).all()

# 3. Comparar los ensambles con los mismos pesos.
parte_compartida_svm = sum(
    pesos_base_svm[nombre] * oof_base_svm[nombre]
    for nombre in ("stacking", "lr_palabras", "est_lr")
)

proba_original_svm = (
    parte_compartida_svm
    + 0.30 * oof_base_svm["estructural_original"]
)
proba_nueva_svm = parte_compartida_svm + 0.30 * oof_svm

pred_original_svm = CLASES[proba_original_svm.argmax(axis=1)]
pred_nueva_svm = CLASES[proba_nueva_svm.argmax(axis=1)]

correcto_original_svm = pred_original_svm == y
correcto_nuevo_svm = pred_nueva_svm == y

resumen_svm = pd.DataFrame([{
    "semilla": SEMILLA_SVM,
    "estructural_original_solo": np.mean(
        CLASES[
            oof_base_svm["estructural_original"].argmax(axis=1)
        ] == y
    ),
    "svm_solo": np.mean(CLASES[oof_svm.argmax(axis=1)] == y),
    "ensamble_original": correcto_original_svm.mean(),
    "ensamble_svm": correcto_nuevo_svm.mean(),
    "mejora_pp": 100 * (
        correcto_nuevo_svm.mean() - correcto_original_svm.mean()
    ),
    "corregidos": int(
        (~correcto_original_svm & correcto_nuevo_svm).sum()
    ),
    "introducidos": int(
        (correcto_original_svm & ~correcto_nuevo_svm).sum()
    ),
}]).set_index("semilla")

pd.DataFrame(resultados_folds_svm).to_csv(
    carpeta_svm / "resultados_por_fold.csv",
    index=False,
)
resumen_svm.to_csv(carpeta_svm / "comparacion_ensamble.csv")

display(resumen_svm.round(6))

print(
    "Tiempo total:",
    round((time.perf_counter() - inicio_svm) / 60, 1),
    "minutos",
)
print("Resultados guardados en:", carpeta_svm)


Entrenando SVM: fold 1/5...
Accuracy SVM: 0.9117 | Tiempo: 9.3 s

Entrenando SVM: fold 2/5...
Accuracy SVM: 0.9017 | Tiempo: 9.0 s

Entrenando SVM: fold 3/5...
Accuracy SVM: 0.9117 | Tiempo: 9.1 s

Entrenando SVM: fold 4/5...
Accuracy SVM: 0.9196 | Tiempo: 9.8 s

Entrenando SVM: fold 5/5...
Accuracy SVM: 0.9067 | Tiempo: 10.2 s


,estructural_original_solo,svm_solo,ensamble_original,ensamble_svm,mejora_pp,corregidos,introducidos
semilla,,,,,,,
42,0.9075,0.91025,0.9165,0.91525,-0.125,192,207


Tiempo total: 0.8 minutos
Resultados guardados en: /Users/alejandrobenavidesrubio/MLP1/results/svm_estructural_20261006_143408


### Resultado del SVM estructural

El SVM RBF superó al componente estructural original de forma aislada
(0.91025 frente a 0.90750), pero su sustitución dentro del ensamble
redujo el accuracy de 0.91650 a 0.91525.

Corrigió 192 errores e introdujo 207. Se conserva el componente original
y no se continúa con la segunda partición.

## 10. Diagnóstico de desacuerdos del componente estructural

Comparamos la predicción del componente estructural con la combinación
de stacking, LR por bloques y est_lr.

La combinación de los otros tres conserva sus proporciones relativas:
0.20, 0.30 y 0.20, normalizadas para sumar 1.

Analizamos por separado los casos de acuerdo y desacuerdo. Dentro de los
desacuerdos también identificamos aquellos donde los tres modelos
alternativos coinciden entre sí.

Este análisis es descriptivo. No seleccionamos umbrales ni modificamos
las predicciones a partir de los resultados.

In [14]:
resumen_desacuerdos = []
tablas_desacuerdos = {}

pesos_alternativos = {
    "stacking": 0.20,
    "lr_palabras": 0.30,
    "est_lr": 0.20,
}

for semilla in (42, 2024):
    probabilidades = {}

    for nombre in [*pesos_alternativos, "estructural_original"]:
        matriz = np.full((len(y), len(CLASES)), np.nan)
        cobertura = np.zeros(len(y), dtype=int)

        archivos = sorted(CARPETA_OOF_BASE.glob(
            f"semilla_{semilla}_fold_*_{nombre}.npz"
        ))
        assert len(archivos) == 5

        for ruta in archivos:
            with np.load(ruta, allow_pickle=False) as archivo:
                indices = archivo["indices"]

                assert np.array_equal(
                    archivo["ids"],
                    train_df["id"].to_numpy()[indices],
                )
                assert np.array_equal(archivo["clases"], CLASES)

                matriz[indices] = archivo["probabilidades"]
                cobertura[indices] += 1

        assert (cobertura == 1).all()
        assert np.isfinite(matriz).all()
        assert np.allclose(matriz.sum(axis=1), 1)

        probabilidades[nombre] = matriz

    proba_otros = sum(
        peso * probabilidades[nombre]
        for nombre, peso in pesos_alternativos.items()
    ) / sum(pesos_alternativos.values())

    proba_estructural = probabilidades["estructural_original"]
    proba_ensamble = 0.70 * proba_otros + 0.30 * proba_estructural

    pred_otros = CLASES[proba_otros.argmax(axis=1)]
    pred_estructural = CLASES[proba_estructural.argmax(axis=1)]
    pred_ensamble = CLASES[proba_ensamble.argmax(axis=1)]

    votos_otros = np.column_stack([
        probabilidades[nombre].argmax(axis=1)
        for nombre in pesos_alternativos
    ])
    unanimidad_otros = (
        votos_otros == votos_otros[:, [0]]
    ).all(axis=1)

    desacuerdo = pred_otros != pred_estructural

    grupo = np.full(len(y), "acuerdo", dtype=object)
    grupo[desacuerdo & ~unanimidad_otros] = (
        "desacuerdo / otros divididos"
    )
    grupo[desacuerdo & unanimidad_otros] = (
        "desacuerdo / otros unanimes"
    )

    tabla = pd.DataFrame({
        "id": train_df["id"].to_numpy(),
        "real": y,
        "grupo": grupo,
        "pred_estructural": pred_estructural,
        "pred_otros": pred_otros,
        "pred_ensamble": pred_ensamble,
        "acierto_estructural": pred_estructural == y,
        "acierto_otros": pred_otros == y,
        "acierto_ensamble": pred_ensamble == y,
    })

    tablas_desacuerdos[semilla] = tabla

    resumen = (
        tabla.groupby("grupo")
        .agg(
            resenas=("id", "size"),
            accuracy_estructural=("acierto_estructural", "mean"),
            accuracy_otros=("acierto_otros", "mean"),
            accuracy_ensamble=("acierto_ensamble", "mean"),
        )
        .reset_index()
    )
    resumen["semilla"] = semilla
    resumen_desacuerdos.append(resumen)

resumen_desacuerdos = (
    pd.concat(resumen_desacuerdos, ignore_index=True)
    .set_index(["semilla", "grupo"])
)

display(resumen_desacuerdos.round(4))

resenas  accuracy_estructural  \
semilla grupo                                                         
42      acuerdo                         11010                0.9466   
        desacuerdo / otros divididos      640                0.5219   
        desacuerdo / otros unanimes       350                0.3829   
2024    acuerdo                         10984                0.9487   
        desacuerdo / otros divididos      665                0.5338   
        desacuerdo / otros unanimes       351                0.3989   

                                      accuracy_otros  accuracy_ensamble  
semilla grupo                                                            
42      acuerdo                               0.9466             0.9466  
        desacuerdo / otros divididos          0.4781             0.5641  
        desacuerdo / otros unanimes           0.6143             0.6143  
2024    acuerdo                               0.9487             0.9487  
        desacuerdo / otros divididos          0.4632             0.5504  
        desacuerdo / otros unanimes           0.6011             0.6068

### Conclusión del diagnóstico de desacuerdos

En los grupos analizados, el ensamble iguala o supera la accuracy
del componente estructural y de la combinación de los otros tres modelos.

La unanimidad de los modelos alternativos identifica casos donde el
estructural rinde peor, pero el ensamble ya aprovecha esa información.

No se implementa una regla de sustitución basada únicamente en el
desacuerdo o la unanimidad. Igualar accuracy no implica necesariamente
producir las mismas predicciones.

## 11. Experimento: ponderación de clases en el especialista

Comparamos dos versiones del especialista:

- Regresión logística sin ponderación de clases.
- La misma regresión logística con class_weight="balanced".

Conservamos la representación TF-IDF, C=1, las reglas de selección
de candidatas y las particiones externas.

Los pesos de las clases se calculan únicamente con el entrenamiento
de cada fold. No se generan ejemplos sintéticos.

Comparamos ambas versiones en las candidatas y medimos el efecto
de sustituir al ensamble en ese grupo. El resto de las predicciones
del ensamble permanece intacto.

Esta prueba es exploratoria y no constituye un experimento con SMOTE.

In [15]:
from datetime import datetime
from sklearn.model_selection import StratifiedKFold
import time

pesos_base_balanceo = {
    "stacking": 0.20,
    "lr_palabras": 0.30,
    "estructural_original": 0.30,
    "est_lr": 0.20,
}

variantes_balanceo = {
    "sin_ponderar": None,
    "balanced": "balanced",
}

marca_balanceo = datetime.now().strftime("%Y%m%d_%H%M%S")
carpeta_balanceo = (
    ROOT / "results" / f"especialista_balanceado_{marca_balanceo}"
)
carpeta_balanceo.mkdir(parents=True, exist_ok=False)

resultados_balanceo = []
oof_balanceo = {}

inicio_balanceo = time.perf_counter()

for semilla in (42, 2024):
    folds = list(
        StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=semilla,
        ).split(X, y)
    )

    # Recuperar las probabilidades del ensamble original.
    proba_base = np.zeros((len(y), len(CLASES)))

    for nombre, peso in pesos_base_balanceo.items():
        matriz = np.full((len(y), len(CLASES)), np.nan)

        for fold, (_, va_idx) in enumerate(folds, start=1):
            ruta = CARPETA_OOF_BASE / (
                f"semilla_{semilla}_fold_{fold}_{nombre}.npz"
            )

            with np.load(ruta, allow_pickle=False) as archivo:
                assert np.array_equal(archivo["indices"], va_idx)
                assert np.array_equal(
                    archivo["ids"],
                    train_df["id"].to_numpy()[va_idx],
                )
                assert np.array_equal(archivo["clases"], CLASES)

                matriz[va_idx] = archivo["probabilidades"]

        assert np.isfinite(matriz).all()
        assert np.allclose(matriz.sum(axis=1), 1)
        proba_base += peso * matriz

    pred_base = CLASES[proba_base.argmax(axis=1)]

    proba_variantes = {
        nombre: np.full((len(y), len(CLASES)), np.nan)
        for nombre in variantes_balanceo
    }
    cobertura = np.zeros(len(y), dtype=int)

    print(f"\n=== Semilla {semilla} ===", flush=True)

    for fold, (tr_idx, va_idx) in enumerate(folds, start=1):
        entrenamiento = especialista_df.loc[
            especialista_df.index.isin(tr_idx)
            & especialista_df["objetivo"].isin(["primera", "ultima"])
        ]

        # La selección de validación depende del texto, no de su etiqueta.
        validacion = especialista_df.loc[
            especialista_df.index.isin(va_idx)
        ]

        assert set(entrenamiento.index).isdisjoint(validacion.index)
        assert entrenamiento["objetivo"].nunique() == 2

        indices = validacion.index.to_numpy()
        if len(indices) == 0:
            continue

        mensajes = []

        for nombre, ponderacion in variantes_balanceo.items():
            candidato = crear_especialista()
            candidato.set_params(
                clasificador__class_weight=ponderacion
            )

            candidato.fit(
                entrenamiento[COLUMNAS_ESPECIALISTA],
                entrenamiento["objetivo"],
            )

            proba_posicion = candidato.predict_proba(
                validacion[COLUMNAS_ESPECIALISTA]
            )

            # Convertir primera/ultima a negativo/neutral/positivo.
            proba_sentimiento = np.zeros((len(indices), len(CLASES)))

            for posicion in ("primera", "ultima"):
                columna = list(candidato.classes_).index(posicion)
                etiquetas = validacion[
                    f"etiqueta_{posicion}"
                ].to_numpy()

                columnas_destino = np.array([
                    list(CLASES).index(etiqueta)
                    for etiqueta in etiquetas
                ])

                proba_sentimiento[
                    np.arange(len(indices)), columnas_destino
                ] += proba_posicion[:, columna]

            assert np.isfinite(proba_sentimiento).all()
            assert np.allclose(proba_sentimiento.sum(axis=1), 1)

            proba_variantes[nombre][indices] = proba_sentimiento

            # Mantener la decisión directa del especialista.
            eleccion = candidato.predict(
                validacion[COLUMNAS_ESPECIALISTA]
            )
            pred_fold = np.where(
                eleccion == "primera",
                validacion["etiqueta_primera"].to_numpy(),
                validacion["etiqueta_ultima"].to_numpy(),
            )

            # Guardar decisiones explícitas evita ambigüedades en empates.
            if fold == 1:
                pass

            mensajes.append(
                f"{nombre}: {np.mean(pred_fold == y[indices]):.4f}"
            )

            np.savez_compressed(
                carpeta_balanceo / (
                    f"semilla_{semilla}_fold_{fold}_{nombre}.npz"
                ),
                indices=indices,
                ids=train_df["id"].to_numpy()[indices],
                clases=CLASES,
                probabilidades=proba_sentimiento,
                predicciones=pred_fold.astype(str),
            )

        cobertura[indices] += 1
        print(
            f"Fold {fold}/5 | " + " | ".join(mensajes),
            flush=True,
        )

    indices_grupo = especialista_df.index.to_numpy()
    assert (cobertura[indices_grupo] == 1).all()

    predicciones_variantes = {}

    for nombre in variantes_balanceo:
        pred_total = pred_base.copy()

        for fold in range(1, 6):
            ruta = carpeta_balanceo / (
                f"semilla_{semilla}_fold_{fold}_{nombre}.npz"
            )
            if ruta.exists():
                with np.load(ruta, allow_pickle=False) as archivo:
                    pred_total[archivo["indices"]] = archivo["predicciones"]

        predicciones_variantes[nombre] = pred_total

    pred_sin = predicciones_variantes["sin_ponderar"]
    pred_bal = predicciones_variantes["balanced"]

    correcto_sin = pred_sin == y
    correcto_bal = pred_bal == y

    resultados_balanceo.append({
        "semilla": semilla,
        "base_grupo": np.mean(pred_base[indices_grupo] == y[indices_grupo]),
        "sin_ponderar_grupo": correcto_sin[indices_grupo].mean(),
        "balanced_grupo": correcto_bal[indices_grupo].mean(),
        "base_global": np.mean(pred_base == y),
        "sin_ponderar_global": correcto_sin.mean(),
        "balanced_global": correcto_bal.mean(),
        "mejora_vs_sin_pp": 100 * (
            correcto_bal.mean() - correcto_sin.mean()
        ),
        "mejora_vs_base_pp": 100 * (
            correcto_bal.mean() - np.mean(pred_base == y)
        ),
        "corregidos_vs_sin": int((~correcto_sin & correcto_bal).sum()),
        "introducidos_vs_sin": int((correcto_sin & ~correcto_bal).sum()),
    })

    oof_balanceo[semilla] = {
        "base": proba_base,
        "indices_candidatas": indices_grupo,
        **{
            nombre: probabilidades[indices_grupo]
            for nombre, probabilidades in proba_variantes.items()
        },
    }

resumen_balanceo = pd.DataFrame(
    resultados_balanceo
).set_index("semilla")

resumen_balanceo.to_csv(carpeta_balanceo / "resultados.csv")
display(resumen_balanceo.round(6))

print(
    "Tiempo total:",
    round((time.perf_counter() - inicio_balanceo) / 60, 1),
    "minutos",
)


=== Semilla 42 ===
Fold 1/5 | sin_ponderar: 0.5025 | balanced: 0.5025
Fold 2/5 | sin_ponderar: 0.5492 | balanced: 0.5622
Fold 3/5 | sin_ponderar: 0.5131 | balanced: 0.5105
Fold 4/5 | sin_ponderar: 0.5519 | balanced: 0.5437
Fold 5/5 | sin_ponderar: 0.4924 | balanced: 0.5051

=== Semilla 2024 ===
Fold 1/5 | sin_ponderar: 0.5079 | balanced: 0.5105
Fold 2/5 | sin_ponderar: 0.5718 | balanced: 0.5718
Fold 3/5 | sin_ponderar: 0.4951 | balanced: 0.5074
Fold 4/5 | sin_ponderar: 0.5201 | balanced: 0.5050
Fold 5/5 | sin_ponderar: 0.4960 | balanced: 0.4826


,base_grupo,sin_ponderar_grupo,balanced_grupo,base_global,sin_ponderar_global,balanced_global,mejora_vs_sin_pp,mejora_vs_base_pp,corregidos_vs_sin,introducidos_vs_sin
semilla,,,,,,,,,,
42,0.536826,0.521266,0.524378,0.916500,0.914000,0.9145,0.050000,-0.200000,31,25
2024,0.534232,0.517635,0.515041,0.916583,0.913917,0.9135,-0.041667,-0.308333,31,36


Tiempo total: 0.1 minutos


### Conclusión de la ponderación de clases

class_weight="balanced" produjo una pequeña mejora frente al especialista
sin ponderar con la semilla 42, pero empeoró con la semilla 2024.

En ambas particiones, sustituir al ensamble por el especialista balanceado
redujo el accuracy global: −0.200 y −0.308 puntos porcentuales.

No se adopta esta configuración ni se genera un submission.
Este experimento evalúa ponderación de clases, no SMOTE.

In [16]:
print("Modelos disponibles en memoria:")

for nombre in (
    "modelo_final",
    "modelo_ensamble_regularizado",
    "modelo_ensamble_lr_final",
):
    objeto = globals().get(nombre)

    if objeto is not None:
        print(f"\n{nombre}: {type(objeto).__name__}")

        if hasattr(objeto, "named_estimators_"):
            print("Componentes:", list(objeto.named_estimators_))
            print("Pesos:", objeto.weights)

            estructural = objeto.named_estimators_.get("estructural")
            if estructural is not None:
                print("Componente estructural:", estructural)

print("\nArchivos de modelos candidatos:")

rutas_candidatas = {
    *ROOT.rglob("*v3b*.joblib"),
    *ROOT.rglob("*ensamble_estructural_regularizado*.joblib"),
}

for ruta in sorted(rutas_candidatas):
    print(ruta.relative_to(ROOT))

if not rutas_candidatas:
    print("No se encontraron esos modelos guardados.")

Modelos disponibles en memoria:

Archivos de modelos candidatos:
models/modelo_ensamble_estructural_regularizado_20261005_221405.joblib
models/modelo_ensamble_estructural_regularizado_20261006_090904.joblib


In [17]:
import ast
import json
import joblib

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline

# 1. Recuperar las definiciones de Bloques y sus conectores.
# Las definiciones de RasgosPlantilla y est_lr ya se cargaron
# durante los experimentos anteriores de este notebook.
with NOTEBOOK_ORIGINAL.open(encoding="utf-8") as archivo:
    fuente_recuperacion = json.load(archivo)

seleccion_por_celda = {
    17: {"FUERTES", "DEBILES", "oraciones"},
    21: {
        "_regex_conectores",
        "RE_FUERTE",
        "RE_DEBIL",
        "RE_ADVERSATIVO",
        "desde_ultimo",
        "Bloques",
    },
}

for numero, nombres_permitidos in seleccion_por_celda.items():
    codigo = "".join(
        fuente_recuperacion["cells"][numero - 1]["source"]
    )
    estructura = ast.parse(codigo)
    nodos = []
    encontrados = set()

    for nodo in estructura.body:
        if isinstance(nodo, (ast.FunctionDef, ast.ClassDef)):
            nombres = {nodo.name}

        elif isinstance(nodo, ast.Assign):
            nombres = {
                elemento.id
                for destino in nodo.targets
                for elemento in ast.walk(destino)
                if isinstance(elemento, ast.Name)
            }

        else:
            continue

        if nombres and nombres.issubset(nombres_permitidos):
            nodos.append(nodo)
            encontrados.update(nombres)

    assert encontrados == nombres_permitidos, (
        f"La celda {numero} cambió; revisar sus definiciones."
    )

    modulo = ast.Module(body=nodos, type_ignores=[])
    ast.fix_missing_locations(modulo)
    exec(
        compile(modulo, f"definiciones_celda_{numero}", "exec"),
        globals(),
    )

# 2. Cargar el ensamble guardado.
ruta_base_95 = (
    ROOT / "models"
    / "modelo_ensamble_estructural_regularizado_20261005_221405.joblib"
)
assert ruta_base_95.exists()

ensamble_base_95 = joblib.load(ruta_base_95)

nombres_base_95 = [
    nombre for nombre, _ in ensamble_base_95.estimators
]
assert nombres_base_95 == [
    "stacking", "lr_palabras", "estructural", "est_lr"
]
assert np.allclose(
    ensamble_base_95.weights,
    [0.20, 0.30, 0.30, 0.20],
)
assert np.array_equal(ensamble_base_95.classes_, CLASES)

# 3. Restaurar el componente estructural ORIGINAL.
# VotingClassifier entrena internamente con etiquetas numéricas.
estructural_original_95 = Pipeline([
    (
        "rasgos",
        RasgosPlantilla(modo_polaridad="aprendida"),
    ),
    (
        "clasificador",
        HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=200,
            max_leaf_nodes=15,
            min_samples_leaf=20,
            l2_regularization=0,
            early_stopping=False,
            random_state=0,
        ),
    ),
])

print("Entrenando el estructural original...", flush=True)

estructural_original_95.fit(
    X,
    ensamble_base_95.le_.transform(y),
)

indice_estructural = nombres_base_95.index("estructural")
ensamble_base_95.estimators_[
    indice_estructural
] = estructural_original_95

ensamble_base_95.named_estimators_[
    "estructural"
] = estructural_original_95

ensamble_base_95.estimators = [
    (
        nombre,
        estructural_original_95
        if nombre == "estructural" else estimador,
    )
    for nombre, estimador in ensamble_base_95.estimators
]

# 4. Entrenar el especialista SIN balanceo:
# es la configuración evaluada en la mezcla 95–5.
entrenamiento_final_95 = especialista_df.loc[
    especialista_df["objetivo"].isin(["primera", "ultima"])
]

especialista_final_95 = crear_especialista()
especialista_final_95.set_params(
    clasificador__class_weight=None
)

print("Entrenando el especialista final...", flush=True)

especialista_final_95.fit(
    entrenamiento_final_95[COLUMNAS_ESPECIALISTA],
    entrenamiento_final_95["objetivo"],
)

print("\nEnsamble original restaurado.")
print("Especialista final entrenado.")
print("Reseñas del especialista:", len(entrenamiento_final_95))
print("Todavía no se han generado predicciones de Kaggle.")

Entrenando el estructural original...
Entrenando el especialista final...

Ensamble original restaurado.
Especialista final entrenado.
Reseñas del especialista: 1928
Todavía no se han generado predicciones de Kaggle.


In [18]:
def predecir_mezcla_95(textos, ensamble, especialista):
    textos = np.asarray(textos)
    clases = np.asarray(ensamble.classes_)

    probabilidades = ensamble.predict_proba(textos).copy()
    filas_candidatas = []
    indices_candidatas = []

    for indice, texto in enumerate(textos):
        par = extraer_par_opiniones(texto)

        if par is None:
            continue

        primera = par["primera"]
        ultima = par["ultima"]

        # Mismo criterio utilizado en validación.
        if es_adjetival(primera) or es_adjetival(ultima):
            continue

        indices_candidatas.append(indice)
        filas_candidatas.append({
            "oracion_primera": primera["oracion_texto"],
            "oracion_ultima": ultima["oracion_texto"],
            "texto_completo": normalizar(texto),
            "conector_ultima": ultima["conector"] or "(ninguno)",
            "etiqueta_primera": (
                "positivo" if par["signo_primera"] == 1 else "negativo"
            ),
            "etiqueta_ultima": (
                "positivo" if par["signo_ultima"] == 1 else "negativo"
            ),
        })

    if filas_candidatas:
        candidatas = pd.DataFrame(filas_candidatas)
        proba_posicion = especialista.predict_proba(
            candidatas[COLUMNAS_ESPECIALISTA]
        )

        proba_especialista = np.zeros(
            (len(candidatas), len(clases))
        )

        for posicion in ("primera", "ultima"):
            columna = list(especialista.classes_).index(posicion)

            destinos = np.array([
                list(clases).index(etiqueta)
                for etiqueta in candidatas[f"etiqueta_{posicion}"]
            ])

            proba_especialista[
                np.arange(len(candidatas)), destinos
            ] += proba_posicion[:, columna]

        indices = np.asarray(indices_candidatas)
        probabilidades[indices] = (
            0.95 * probabilidades[indices]
            + 0.05 * proba_especialista
        )

    assert np.isfinite(probabilidades).all()
    assert np.allclose(probabilidades.sum(axis=1), 1)

    predicciones = clases[probabilidades.argmax(axis=1)]
    return predicciones, probabilidades, np.asarray(
        indices_candidatas, dtype=int
    )


# Cargar y validar los archivos de evaluación.
eval_95 = pd.read_csv(DATA_DIR / "eval.csv")
plantilla_95 = pd.read_csv(DATA_DIR / "sample_submission.csv")

assert list(plantilla_95.columns) == ["id", "answer"]
assert eval_95[["id", "text"]].notna().all().all()
assert eval_95["text"].str.strip().ne("").all()
assert plantilla_95["id"].notna().all()
assert eval_95["id"].is_unique
assert plantilla_95["id"].is_unique
assert set(eval_95["id"]) == set(plantilla_95["id"])

pred_95, proba_95, candidatas_95 = predecir_mezcla_95(
    eval_95["text"].to_numpy(),
    ensamble_base_95,
    especialista_final_95,
)

pred_base_95 = ensamble_base_95.predict(
    eval_95["text"].to_numpy()
)

# Comprobar que fuera de las candidatas no cambia nada.
mascara_95 = np.zeros(len(eval_95), dtype=bool)
mascara_95[candidatas_95] = True

assert np.array_equal(
    pred_95[~mascara_95],
    pred_base_95[~mascara_95],
)

submission_95 = plantilla_95.copy()
submission_95["answer"] = submission_95["id"].map(
    pd.Series(pred_95, index=eval_95["id"])
)

assert submission_95["answer"].notna().all()
assert set(submission_95["answer"]).issubset(CLASES)

print("Reseñas de evaluación:", len(eval_95))
print("Candidatas al especialista:", len(candidatas_95))
print(
    "Predicciones distintas respecto al ensamble base:",
    int((pred_95 != pred_base_95).sum()),
)

display(submission_95.head())

Reseñas de evaluación: 3000
Candidatas al especialista: 486
Predicciones distintas respecto al ensamble base: 9


,id,answer
0,0,neutral
1,1,positivo
2,2,neutral
3,3,neutral
4,4,neutral


In [19]:
from datetime import datetime
import joblib

marca_95 = datetime.now().strftime("%Y%m%d_%H%M%S")
nombre_95 = f"ensamble_especialista_95_5_{marca_95}"

ruta_modelo_95 = ROOT / "models" / f"modelo_{nombre_95}.joblib"
ruta_csv_95 = ROOT / "submissions" / f"submission_{nombre_95}.csv"

for ruta in (ruta_modelo_95, ruta_csv_95):
    ruta.parent.mkdir(parents=True, exist_ok=True)
    assert not ruta.exists(), f"El archivo ya existe: {ruta}"

# Guardar ambos modelos y la configuración de la mezcla.
paquete_95 = {
    "ensamble": ensamble_base_95,
    "especialista": especialista_final_95,
    "peso_especialista": 0.05,
    "criterio": "exactamente dos frases fijas de polaridad opuesta",
    "columnas_especialista": COLUMNAS_ESPECIALISTA,
    "clases": CLASES,
    "versiones": {
        "python": sys.version.split()[0],
        "sklearn": sklearn.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
}

joblib.dump(paquete_95, ruta_modelo_95, compress=3)
submission_95.to_csv(ruta_csv_95, index=False)

# Verificar la recarga.
recargado_95 = joblib.load(ruta_modelo_95)
assert recargado_95["peso_especialista"] == 0.05

pred_recargada_95, proba_recargada_95, _ = predecir_mezcla_95(
    eval_95["text"].to_numpy(),
    recargado_95["ensamble"],
    recargado_95["especialista"],
)

assert np.array_equal(pred_95, pred_recargada_95)
assert np.allclose(proba_95, proba_recargada_95)

csv_guardado_95 = pd.read_csv(ruta_csv_95)
esperadas_95 = csv_guardado_95["id"].map(
    pd.Series(pred_recargada_95, index=eval_95["id"])
)

assert list(csv_guardado_95.columns) == ["id", "answer"]
assert np.array_equal(
    csv_guardado_95["id"].to_numpy(),
    plantilla_95["id"].to_numpy(),
)
assert np.array_equal(
    csv_guardado_95["answer"].to_numpy(),
    esperadas_95.to_numpy(),
)

print("OK: los modelos recargados reproducen el CSV.")
print("\nARCHIVO PARA SUBIR A KAGGLE:")
print(ruta_csv_95)
print("\nModelos guardados en:")
print(ruta_modelo_95)

OK: los modelos recargados reproducen el CSV.

ARCHIVO PARA SUBIR A KAGGLE:
/Users/alejandrobenavidesrubio/MLP1/submissions/submission_ensamble_especialista_95_5_20261006_145230.csv

Modelos guardados en:
/Users/alejandrobenavidesrubio/MLP1/models/modelo_ensamble_especialista_95_5_20261006_145230.joblib


### Resultado de la mezcla 95–5 en Kaggle

El submission obtuvo un Public Score de 0.90555.

El especialista se aplicó a 486 de las 3.000 reseñas y cambió
9 etiquetas respecto al ensamble base reconstruido.

No superó el mejor resultado reportado del equipo, aproximadamente
0.9077. Se conserva el mejor envío anterior.

Este resultado no demuestra una mejora del especialista, en línea
con la falta de consistencia observada entre las dos particiones
de validación.